# Init

## Import

In [1]:
import warnings
import random
import time
import os
import re
import torch
import shutil
import multiprocessing
import subprocess
import cv2

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim

import numpy as np
import plotly.graph_objects as go

from torch.amp import GradScaler
from torch.amp import autocast
from torch.optim.adamax import Adamax
from torch.utils.data import random_split
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import Subset
from torch.utils.data import ConcatDataset
from torch.autograd import Variable

from torch.nn import Parameter

from PIL import Image
from torchvision import transforms

from typing import Literal
from typing import Dict

from pathlib import Path
from tqdm.notebook import tqdm
from os.path import join
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize

from google.colab import drive

warnings.filterwarnings("ignore")
# warnings.filterwarnings("ignore", module="tqdm")

## Utils

In [2]:
import sys

class HyperparameterCollector:
    def __init__(self):
        self._params = {}

    def extend(self, *var_names):
        frame = sys._getframe(1)
        local_vars = frame.f_locals

        for name in var_names:
            if name in local_vars and not name.startswith('_'):
                self._params[name] = local_vars[name]

        return self

    def to_dict(self):
        return self._params.copy()

# hp = HyperparameterCollector()
# hp_train_metrics = HyperparameterCollector()

hp = dict()

In [3]:
def title(text):
    print("\n" + "="*50)
    print(text)
    print("="*50)


In [4]:
from functools import wraps

def timer_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        execution_time = end_time - start_time
        print(f"Функция '{func.__name__}' выполнилась за {execution_time:0f} секунд")
        return result
    return wrapper



@timer_decorator
def cmd_zip(folder_path, zip_path, verbose=False):
    subprocess.run(
        f"zip -j -q {zip_path} {folder_path}/*",
        shell=True,
    )
    if verbose:
        print('zip done')

@timer_decorator
def cmd_unzip(folder_path, zip_path, verbose=False):
    subprocess.run(
        f"unzip -q -d {folder_path} -o {zip_path}",
        shell=True,
    )
    if verbose:
        print('unzip done')

@timer_decorator
def cmd_cp(source_zip_path, dest_zip_path, verbose=False):
    subprocess.run(
        f"cp {source_zip_path} {dest_zip_path}",
        shell=True
    )
    if verbose:
        print('cp done')

@timer_decorator
def cmd_rm_rf(folder_path, verbose=False):
    subprocess.run(
        f"rm -rf {folder_path}",
        shell=True
    )
    if verbose:
        print('rm done')

## Env

In [5]:
# Монтируем диск:
drive.mount('/content/drive')

# Настраиваем окружение:
colab   = True
use_amp = True
cuda    = torch.cuda.is_available()
device  = torch.device('cuda' if cuda else 'cpu')
cores   = multiprocessing.cpu_count()

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.allow_tf32 = True

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
    capture_output=True,
    text=True
)
gpu_name = result.stdout.strip()

title("Параметры среды:")

print(f'Running in Google Colab: {colab}')
print(f'Use Mixed precision: {use_amp}')
print(f'Use GPU: {cuda}')
print(f"GPU Model: {gpu_name}")
print(f'Device: {device}')
print(f"Количество CPU-ядер: {cores}")

hp['env'] = dict(
    colab=colab,
    use_amp=use_amp,
    cude=cuda,
    device=device,
    cores=cores,
    gpu_name=gpu_name,
)

srm_path = '/content/drive/MyDrive/diplom/SRM_Kernels.npy'
SRM_npy = np.load(srm_path)


In [6]:
TEST_loader_kwargs = dict(
    num_workers = 12,
    prefetch_factor = 16,
    persistent_workers = True,
    shuffle = False,
    pin_memory = True,
    drop_last = False,
)

# [MODULE] Net

## Common

### Blocks

In [7]:
class SRMConv2d(nn.Module):

    def __init__(self, stride=1, padding=0):
        super(SRMConv2d, self).__init__()
        self.in_channels = 1
        self.out_channels = 30
        self.kernel_size = (5, 5)
        if isinstance(stride, int):
            self.stride = (stride, stride)
        else:
            self.stride = stride
        if isinstance(padding, int):
            self.padding = (padding, padding)
        else:
            self.padding = padding

        self.dilation = (1, 1)
        self.transpose = False
        self.output_padding = (0,)
        self.groups = 1
        self.weight = Parameter(torch.Tensor(30, 1, 5, 5), requires_grad=True)
        self.bias = Parameter(torch.Tensor(30), requires_grad=True)
        self.reset_parameters()

    def reset_parameters(self):
        self.weight.data.numpy()[:] = SRM_npy
        self.bias.data.zero_()

    def forward(self, input):
        return F.conv2d(input, self.weight, self.bias, self.stride, self.padding,
                        self.dilation, self.groups)



## SiaStegNet

### Loss

In [8]:
class ContrastiveLoss(nn.Module):

    def __init__(self, margin=1.25):  # margin=2
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, output1, output2, label):
        label = label.to(torch.float32)
        euclidean_distance = F.pairwise_distance(output1, output2)
        loss_contrastive = torch.mean(
            (1 - label) * torch.pow(euclidean_distance, 2) +
            label * torch.pow(torch.clamp(self.margin - euclidean_distance, min=0.0), 2)
        )

        return loss_contrastive


### Blocks

In [9]:
def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=dilation, groups=groups, bias=False, dilation=dilation)


def conv1x1(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


class BlockA(nn.Module):

    def __init__(self, in_planes, out_planes, norm_layer=None):
        super(BlockA, self).__init__()

        if norm_layer is None:
            norm_layer = nn.BatchNorm2d

        self.conv1 = conv3x3(in_planes, out_planes)
        self.bn1 = norm_layer(out_planes)
        self.conv2 = conv3x3(out_planes, out_planes)
        self.bn2 = norm_layer(out_planes)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += identity
        out = self.relu(out)

        return out


class BlockB(nn.Module):

    def __init__(self, in_planes, out_planes, norm_layer=None):
        super(BlockB, self).__init__()

        if norm_layer is None:
            norm_layer = nn.BatchNorm2d

        self.conv1 = conv3x3(in_planes, out_planes, stride=2)
        self.bn1 = norm_layer(out_planes)
        self.conv2 = conv3x3(out_planes, out_planes)
        self.bn2 = norm_layer(out_planes)
        # self.pool = nn.AvgPool2d(3, stride=2, padding=1)

        self.shortcut_conv = conv1x1(in_planes, out_planes, stride=2)
        self.shortcut_bn = norm_layer(out_planes)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        # out = self.pool(out)

        identity = self.shortcut_conv(identity)
        identity = self.shortcut_bn(identity)

        out += identity
        out = self.relu(out)

        return out


### SiaStegNet (original)

In [10]:
class SiaStegNet(nn.Module):

    def __init__(self, norm_layer=None, zero_init_residual=True, p=0.5):
        super(SiaStegNet, self).__init__()

        self.zero_init_residual = zero_init_residual

        if norm_layer is None:
            norm_layer = nn.BatchNorm2d

        self.srm = SRMConv2d(1, 0)
        self.bn1 = norm_layer(30)

        self.A1 = BlockA(30, 30, norm_layer=norm_layer)
        self.A2 = BlockA(30, 30, norm_layer=norm_layer)
        self.AA = BlockA(30, 30, norm_layer=norm_layer)

        self.B1 = BlockB(30, 30, norm_layer=norm_layer)
        self.B2 = BlockB(30, 64, norm_layer=norm_layer)

        self.B3 = BlockB(30, 64, norm_layer=norm_layer)
        self.A3 = BlockA(64, 64, norm_layer=norm_layer)

        self.B4 = BlockB(64, 128, norm_layer=norm_layer)
        self.A4 = BlockA(128, 128, norm_layer=norm_layer)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.bnfc = nn.BatchNorm1d(128)
        self.relu = nn.ReLU(inplace=True)
        self.fcfusion = nn.Linear(128, 128) #4
        self.fc = nn.Linear(128 * 4 + 1, 2)
        self.dropout = nn.Dropout(p=p)

        self.reset_parameters()

    def reset_parameters(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                # nn.init.xavier_uniform_(m.weight)
                # nn.init.constant_(m.bias, 0.2)
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01)

        if self.zero_init_residual:
            for m in self.modules():
                if isinstance(m, (BlockA, BlockB)):
                    nn.init.constant_(m.bn2.weight, 0)

    def extract_feat(self, x):
        x = x.float()
        out = self.srm(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.A1(out)
        out = self.A2(out)
        out = self.AA(out)

        # out = self.B1(out)
        # out = self.B2(out)

        out = self.B3(out)
        out = self.A3(out)

        out = self.B4(out)
        out = self.A4(out)

        out = self.avgpool(out)
        out = out.view(out.size(0), out.size(1))

        # out = self.relu(out)
        # out = self.bnfc(out)

        return out

    def forward(self, *args):
        feats = torch.stack(
            [self.extract_feat(subarea) for subarea in args], dim=0
        )

        euclidean_distance = F.pairwise_distance(feats[0], feats[1], eps=1e-6, keepdim=True)

        if feats.shape[0] == 1:
            final_feat = feats.squeeze(dim=0)
        else:
            # feats_sum = feats.sum(dim=0)
            # feats_sub = feats[0] - feats[1]
            feats_mean = feats.mean(dim=0)
            feats_var = feats.var(dim=0)
            feats_min, _ = feats.min(dim=0)
            feats_max, _ = feats.max(dim=0)

            final_feat = torch.cat(
                [euclidean_distance, feats_mean, feats_var, feats_min, feats_max], dim=-1
                #[euclidean_distance, feats_sum, feats_sub, feats_prod, feats_max], dim=-1
            )

        out = self.dropout(final_feat)
        # out = self.fcfusion(out)
        # out = self.relu(out)
        out = self.fc(out)

        return out, feats[0], feats[1]

### SiaStegNet (SE)

In [11]:
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.LeakyReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

In [12]:
class BlockA_w_SE(nn.Module):

    def __init__(self, in_planes, out_planes, norm_layer=None, use_se=False):
        super(BlockA_w_SE, self).__init__()

        if norm_layer is None:
            norm_layer = nn.BatchNorm2d

        self.conv1 = conv3x3(in_planes, out_planes)
        self.bn1 = norm_layer(out_planes)
        self.conv2 = conv3x3(out_planes, out_planes)
        self.bn2 = norm_layer(out_planes)

        self.relu = nn.ReLU(inplace=True)

        self.use_se = use_se
        if self.use_se:
            self.se = SEBlock(out_planes, reduction=16)

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.use_se:
            out = self.se(out)

        out += identity
        out = self.relu(out)

        return out


class BlockB_w_SE(nn.Module):

    def __init__(self, in_planes, out_planes, norm_layer=None, use_se=False):
        super(BlockB_w_SE, self).__init__()

        if norm_layer is None:
            norm_layer = nn.BatchNorm2d

        self.conv1 = conv3x3(in_planes, out_planes, stride=2)
        self.bn1 = norm_layer(out_planes)
        self.conv2 = conv3x3(out_planes, out_planes)
        self.bn2 = norm_layer(out_planes)

        self.shortcut_conv = conv1x1(in_planes, out_planes, stride=2)
        self.shortcut_bn = norm_layer(out_planes)

        self.relu = nn.ReLU(inplace=True)

        self.use_se = use_se
        if self.use_se:
            self.se = SEBlock(out_planes, reduction=16)

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.use_se:
            out = self.se(out)

        identity = self.shortcut_conv(identity)
        identity = self.shortcut_bn(identity)

        out += identity
        out = self.relu(out)

        return out


In [13]:
class SiaStegNet_SE(nn.Module):

    def __init__(self, norm_layer=None, zero_init_residual=True, p=0.5, use_se=True):
        super(SiaStegNet_SE, self).__init__()

        self.zero_init_residual = zero_init_residual

        if norm_layer is None:
            norm_layer = nn.BatchNorm2d

        self.srm = SRMConv2d(1, 0)
        self.bn1 = norm_layer(30)

        self.A1 = BlockA_w_SE(30, 30, norm_layer=norm_layer, use_se=use_se)
        self.A2 = BlockA_w_SE(30, 30, norm_layer=norm_layer, use_se=use_se)
        self.AA = BlockA_w_SE(30, 30, norm_layer=norm_layer, use_se=use_se)

        self.B1 = BlockB_w_SE(30, 30, norm_layer=norm_layer, use_se=use_se)
        self.B2 = BlockB_w_SE(30, 64, norm_layer=norm_layer, use_se=use_se)

        self.B3 = BlockB_w_SE(30, 64, norm_layer=norm_layer, use_se=use_se)
        self.A3 = BlockA_w_SE(64, 64, norm_layer=norm_layer, use_se=use_se)

        self.B4 = BlockB_w_SE(64, 128, norm_layer=norm_layer, use_se=use_se)
        self.A4 = BlockA_w_SE(128, 128, norm_layer=norm_layer, use_se=use_se)


        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.bnfc = nn.BatchNorm1d(128)
        self.relu = nn.ReLU(inplace=True)
        self.fcfusion = nn.Linear(128, 128) #4
        self.fc = nn.Linear(128 * 4 + 1, 2)
        self.dropout = nn.Dropout(p=p)

        self.reset_parameters()



    def reset_parameters(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                # nn.init.xavier_uniform_(m.weight)
                # nn.init.constant_(m.bias, 0.2)
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01)

        if self.zero_init_residual:
            for m in self.modules():
                if isinstance(m, (BlockA, BlockB)):
                    nn.init.constant_(m.bn2.weight, 0)

    def extract_feat(self, x):
        x = x.float()
        out = self.srm(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.A1(out)
        out = self.A2(out)
        out = self.AA(out)

        out = self.B3(out)
        out = self.A3(out)

        out = self.B4(out)
        out = self.A4(out)

        out = self.avgpool(out)
        out = out.view(out.size(0), out.size(1))

        return out

    def forward(self, *args):
        feats = torch.stack(
            [self.extract_feat(subarea) for subarea in args], dim=0
        )

        euclidean_distance = F.pairwise_distance(feats[0], feats[1], eps=1e-6, keepdim=True)

        if feats.shape[0] == 1:
            final_feat = feats.squeeze(dim=0)
        else:
            feats_mean = feats.mean(dim=0)
            feats_var = feats.var(dim=0)
            feats_min, _ = feats.min(dim=0)
            feats_max, _ = feats.max(dim=0)

            final_feat = torch.cat(
                [euclidean_distance, feats_mean, feats_var, feats_min, feats_max], dim=-1
            )

        out = self.dropout(final_feat)
        out = self.fc(out)

        return out, feats[0], feats[1]

### SiaStegNet (SE + CVT)

In [14]:
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)

        # В статье используется LeakyReLU внутри SE блока
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.LeakyReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

In [15]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=2.0, dropout=0.1):
        super(TransformerBlock, self).__init__()

        self.norm1 = nn.LayerNorm(embed_dim, eps=1e-6)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)

        self.norm2 = nn.LayerNorm(embed_dim, eps=1e-6)

        # MLP Block
        hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(), # В коде используется tf.nn.gelu
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        norm_x = self.norm1(x)
        attn_out, _ = self.attn(norm_x, norm_x, norm_x)
        x = x + attn_out

        norm_x = self.norm2(x)
        mlp_out = self.mlp(norm_x)
        x = x + mlp_out

        return x


class CVTTransformer(nn.Module):
    def __init__(self, in_channels, embed_dim=128, patch_size=4, num_layers=4, num_heads=4, mlp_ratio=2.0, reduction=32):
        super(CVTTransformer, self).__init__()

        self.pre_se = SEBlock(in_channels, reduction=reduction)
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size, padding=0)

        self.pos_embed = nn.Parameter(torch.zeros(1, 1000, embed_dim))
        self.dropout = nn.Dropout(0.1)

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout=0.1)
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(embed_dim, eps=1e-6)

    def forward(self, x):
        # Вход: (Batch, Channels, Height, Width)

        x = self.pre_se(x)

        x = self.proj(x)
        B, C, H, W = x.shape

        x = x.flatten(2).transpose(1, 2)
        N = x.shape[1]

        if self.pos_embed.shape[1] < N:
            pad = torch.zeros(B, N - self.pos_embed.shape[1], C, device=x.device)
            pos_embed = torch.cat([self.pos_embed, pad], dim=1)
        else:
            pos_embed = self.pos_embed[:, :N, :]

        x = x + pos_embed
        x = self.dropout(x)

        for blk in self.blocks:
            x = blk(x)

        x = self.norm(x)

        return x # (B, N, C)



In [16]:
class BlockA_w_SE(nn.Module):

    def __init__(self, in_planes, out_planes, norm_layer=None, use_se=False):
        super(BlockA_w_SE, self).__init__()

        if norm_layer is None:
            norm_layer = nn.BatchNorm2d

        self.conv1 = conv3x3(in_planes, out_planes)
        self.bn1 = norm_layer(out_planes)
        self.conv2 = conv3x3(out_planes, out_planes)
        self.bn2 = norm_layer(out_planes)

        self.relu = nn.ReLU(inplace=True)

        self.use_se = use_se
        if self.use_se:
            self.se = SEBlock(out_planes, reduction=16)

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.use_se:
            out = self.se(out)

        out += identity
        out = self.relu(out)

        return out


class BlockB_w_SE(nn.Module):

    def __init__(self, in_planes, out_planes, norm_layer=None, use_se=False):
        super(BlockB_w_SE, self).__init__()

        if norm_layer is None:
            norm_layer = nn.BatchNorm2d

        self.conv1 = conv3x3(in_planes, out_planes, stride=2)
        self.bn1 = norm_layer(out_planes)
        self.conv2 = conv3x3(out_planes, out_planes)
        self.bn2 = norm_layer(out_planes)

        self.shortcut_conv = conv1x1(in_planes, out_planes, stride=2)
        self.shortcut_bn = norm_layer(out_planes)

        self.relu = nn.ReLU(inplace=True)

        self.use_se = use_se
        if self.use_se:
            self.se = SEBlock(out_planes, reduction=16)

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.use_se:
            out = self.se(out)

        identity = self.shortcut_conv(identity)
        identity = self.shortcut_bn(identity)

        out += identity
        out = self.relu(out)

        return out


In [17]:
class SiaStegNet_CVT(nn.Module):

    def __init__(self, norm_layer=None, zero_init_residual=True, p=0.5, use_se=True,
                 embed_dim=128, path_size=2, num_layers=2, num_heads=4, reduction=16):
        super(SiaStegNet_CVT, self).__init__()

        self.zero_init_residual = zero_init_residual

        if norm_layer is None:
            norm_layer = nn.BatchNorm2d

        self.srm = SRMConv2d(1, 0)
        self.bn1 = norm_layer(30)

        self.A1 = BlockA_w_SE(30, 30, norm_layer=norm_layer, use_se=use_se)
        self.A2 = BlockA_w_SE(30, 30, norm_layer=norm_layer, use_se=use_se)
        self.AA = BlockA_w_SE(30, 30, norm_layer=norm_layer, use_se=use_se)

        self.B1 = BlockB_w_SE(30, 30, norm_layer=norm_layer, use_se=use_se)
        self.B2 = BlockB_w_SE(30, 64, norm_layer=norm_layer, use_se=use_se)

        self.B3 = BlockB_w_SE(30, 64, norm_layer=norm_layer, use_se=use_se)
        self.A3 = BlockA_w_SE(64, 64, norm_layer=norm_layer, use_se=use_se)

        self.B4 = BlockB_w_SE(64, 128, norm_layer=norm_layer, use_se=use_se)
        self.A4 = BlockA_w_SE(128, 128, norm_layer=norm_layer, use_se=use_se)

        # Блок внимания:
        self.cvt_transformer = CVTTransformer(
            in_channels=128,
            embed_dim=128,
            patch_size=2,
            num_layers=2,    # Количество слоев трансформера
            num_heads=4,     # Количество голов внимания
            reduction=16
        )


        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.bnfc = nn.BatchNorm1d(128)
        self.relu = nn.ReLU(inplace=True)
        self.fcfusion = nn.Linear(128, 128) #4
        self.fc = nn.Linear(128 * 4 + 1, 2)
        self.dropout = nn.Dropout(p=p)

        self.reset_parameters()



    def reset_parameters(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                # nn.init.xavier_uniform_(m.weight)
                # nn.init.constant_(m.bias, 0.2)
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01)

        if self.zero_init_residual:
            for m in self.modules():
                if isinstance(m, (BlockA_w_SE, BlockB_w_SE)):
                    nn.init.constant_(m.bn2.weight, 0)

    def extract_feat(self, x):
        x = x.float()
        out = self.srm(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.A1(out)
        out = self.A2(out)
        out = self.AA(out)

        out = self.B3(out)
        out = self.A3(out)

        out = self.B4(out)
        out = self.A4(out)

        # Use CVT:
        out = self.cvt_transformer(out)
        out = out.mean(dim=1)
        out = self.bnfc(out)

        # Instead of CVT:
        # out = self.avgpool(out)
        # out = out.view(out.size(0), out.size(1))

        return out

    def forward(self, *args):
        feats = torch.stack(
            [self.extract_feat(subarea) for subarea in args], dim=0
        )

        euclidean_distance = F.pairwise_distance(feats[0], feats[1], eps=1e-6, keepdim=True)

        if feats.shape[0] == 1:
            final_feat = feats.squeeze(dim=0)
        else:
            feats_mean = feats.mean(dim=0)
            feats_var = feats.var(dim=0)
            feats_min, _ = feats.min(dim=0)
            feats_max, _ = feats.max(dim=0)

            final_feat = torch.cat(
                [euclidean_distance, feats_mean, feats_var, feats_min, feats_max], dim=-1
            )

        out = self.dropout(final_feat)
        out = self.fc(out)

        return out, feats[0], feats[1]

### SiaStegNet + FC

In [18]:
class SiaStegNet_FC(nn.Module):

    def __init__(self, norm_layer=None, zero_init_residual=True, p=0.5):
        super(SiaStegNet_FC, self).__init__()

        self.zero_init_residual = zero_init_residual

        if norm_layer is None:
            norm_layer = nn.BatchNorm2d

        self.srm = SRMConv2d(1, 0)
        self.bn1 = norm_layer(30)

        self.A1 = BlockA(30, 30, norm_layer=norm_layer)
        self.A2 = BlockA(30, 30, norm_layer=norm_layer)
        self.AA = BlockA(30, 30, norm_layer=norm_layer)

        self.B1 = BlockB(30, 30, norm_layer=norm_layer)
        self.B2 = BlockB(30, 64, norm_layer=norm_layer)

        self.B3 = BlockB(30, 64, norm_layer=norm_layer)
        self.A3 = BlockA(64, 64, norm_layer=norm_layer)

        self.B4 = BlockB(64, 128, norm_layer=norm_layer)
        self.A4 = BlockA(128, 128, norm_layer=norm_layer)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.bnfc = nn.BatchNorm1d(128)
        self.relu = nn.ReLU(inplace=True)
        self.fcfusion = nn.Linear(128, 128) #4
        self.fc = nn.Linear(128 * 6 + 1, 2)
        self.dropout = nn.Dropout(p=p)

        self.reset_parameters()

    def reset_parameters(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                # nn.init.xavier_uniform_(m.weight)
                # nn.init.constant_(m.bias, 0.2)
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01)

        if self.zero_init_residual:
            for m in self.modules():
                if isinstance(m, (BlockA, BlockB)):
                    nn.init.constant_(m.bn2.weight, 0)

    def extract_feat(self, x):
        x = x.float()
        out = self.srm(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.A1(out)
        out = self.A2(out)
        out = self.AA(out)

        # out = self.B1(out)
        # out = self.B2(out)

        out = self.B3(out)
        out = self.A3(out)

        out = self.B4(out)
        out = self.A4(out)

        out = self.avgpool(out)
        out = out.view(out.size(0), out.size(1))

        # out = self.relu(out)
        # out = self.bnfc(out)

        return out

    def forward(self, *args):
        feats = torch.stack(
            [self.extract_feat(subarea) for subarea in args], dim=0
        )

        euclidean_distance = F.pairwise_distance(feats[0], feats[1], eps=1e-6, keepdim=True)

        if feats.shape[0] == 1:
            final_feat = feats.squeeze(dim=0)
        else:
            feats_sum = feats.sum(dim=0)
            feats_sub = feats[0] - feats[1]
            feats_mean = feats.mean(dim=0)
            feats_var = feats.var(dim=0)
            feats_min, _ = feats.min(dim=0)
            feats_max, _ = feats.max(dim=0)

            final_feat = torch.cat(
                [euclidean_distance, feats_mean, feats_var, feats_min, feats_max, feats_sum, feats_sub], dim=-1
            )

        out = self.dropout(final_feat)
        # out = self.fcfusion(out)
        # out = self.relu(out)
        out = self.fc(out)

        return out, feats[0], feats[1]

### SiaStegNet – 4 nets

In [19]:
class SiaStegNet_4S(nn.Module):

    def __init__(self, norm_layer=None, zero_init_residual=True, p=0.5):
        super(SiaStegNet_4S, self).__init__()

        self.zero_init_residual = zero_init_residual

        if norm_layer is None:
            norm_layer = nn.BatchNorm2d

        self.srm = SRMConv2d(1, 0)
        self.bn1 = norm_layer(30)

        self.A1 = BlockA(30, 30, norm_layer=norm_layer)
        self.A2 = BlockA(30, 30, norm_layer=norm_layer)
        self.AA = BlockA(30, 30, norm_layer=norm_layer)

        self.B1 = BlockB(30, 30, norm_layer=norm_layer)
        self.B2 = BlockB(30, 64, norm_layer=norm_layer)

        self.B3 = BlockB(30, 64, norm_layer=norm_layer)
        self.A3 = BlockA(64, 64, norm_layer=norm_layer)

        self.B4 = BlockB(64, 128, norm_layer=norm_layer)
        self.A4 = BlockA(128, 128, norm_layer=norm_layer)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.bnfc = nn.BatchNorm1d(128)
        self.relu = nn.ReLU(inplace=True)
        self.fcfusion = nn.Linear(128, 128) #4
        self.fc = nn.Linear(128 * 4 + 1, 2)
        self.dropout = nn.Dropout(p=p)

        self.reset_parameters()

    def reset_parameters(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                # nn.init.xavier_uniform_(m.weight)
                # nn.init.constant_(m.bias, 0.2)
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01)

        if self.zero_init_residual:
            for m in self.modules():
                if isinstance(m, (BlockA, BlockB)):
                    nn.init.constant_(m.bn2.weight, 0)

    def extract_feat(self, x):
        x = x.float()
        out = self.srm(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.A1(out)
        out = self.A2(out)
        out = self.AA(out)

        out = self.B3(out)
        out = self.A3(out)

        out = self.B4(out)
        out = self.A4(out)

        out = self.avgpool(out)
        out = out.view(out.size(0), out.size(1))


        return out

    def forward(self, *args):
        feats = torch.stack(
            [self.extract_feat(subarea) for subarea in args], dim=0
        )

        d01 = F.pairwise_distance(feats[0], feats[1], keepdim=True)
        d02 = F.pairwise_distance(feats[0], feats[2], keepdim=True)
        d03 = F.pairwise_distance(feats[0], feats[3], keepdim=True)
        d12 = F.pairwise_distance(feats[1], feats[2], keepdim=True)
        d13 = F.pairwise_distance(feats[1], feats[3], keepdim=True)
        d23 = F.pairwise_distance(feats[2], feats[3], keepdim=True)

        mean_euclidean_distance = (d01 + d02 + d03 + d12 + d13 + d23) / 6.0


        if feats.shape[0] == 1:
            final_feat = feats.squeeze(dim=0)
        else:
            feats_mean = feats.mean(dim=0)
            feats_var = feats.var(dim=0)
            feats_min, _ = feats.min(dim=0)
            feats_max, _ = feats.max(dim=0)

            final_feat = torch.cat(
                [mean_euclidean_distance, feats_mean, feats_var, feats_min, feats_max], dim=-1
            )

        out = self.dropout(final_feat)
        out = self.fc(out)

        return out, feats[0], feats[1]

### SiaStegNet – 4 nets + SE

In [20]:
class SiaStegNet_4S_SE(nn.Module):

    def __init__(self, norm_layer=None, zero_init_residual=True, p=0.5, use_se=True):
        super(SiaStegNet_4S_SE, self).__init__()

        self.zero_init_residual = zero_init_residual

        if norm_layer is None:
            norm_layer = nn.BatchNorm2d

        self.srm = SRMConv2d(1, 0)
        self.bn1 = norm_layer(30)

        self.A1 = BlockA_w_SE(30, 30, norm_layer=norm_layer, use_se=use_se)
        self.A2 = BlockA_w_SE(30, 30, norm_layer=norm_layer, use_se=use_se)
        self.AA = BlockA_w_SE(30, 30, norm_layer=norm_layer, use_se=use_se)

        self.B1 = BlockB_w_SE(30, 30, norm_layer=norm_layer, use_se=use_se)
        self.B2 = BlockB_w_SE(30, 64, norm_layer=norm_layer, use_se=use_se)

        self.B3 = BlockB_w_SE(30, 64, norm_layer=norm_layer, use_se=use_se)
        self.A3 = BlockA_w_SE(64, 64, norm_layer=norm_layer, use_se=use_se)

        self.B4 = BlockB_w_SE(64, 128, norm_layer=norm_layer, use_se=use_se)
        self.A4 = BlockA_w_SE(128, 128, norm_layer=norm_layer, use_se=use_se)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.bnfc = nn.BatchNorm1d(128)
        self.relu = nn.ReLU(inplace=True)
        self.fcfusion = nn.Linear(128, 128) #4
        self.fc = nn.Linear(128 * 4 + 1, 2)
        self.dropout = nn.Dropout(p=p)

        self.reset_parameters()

    def reset_parameters(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                # nn.init.xavier_uniform_(m.weight)
                # nn.init.constant_(m.bias, 0.2)
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01)

        if self.zero_init_residual:
            for m in self.modules():
                if isinstance(m, (BlockA_w_SE, BlockB_w_SE)):
                    nn.init.constant_(m.bn2.weight, 0)

    def extract_feat(self, x):
        x = x.float()
        out = self.srm(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.A1(out)
        out = self.A2(out)
        out = self.AA(out)

        out = self.B3(out)
        out = self.A3(out)

        out = self.B4(out)
        out = self.A4(out)

        out = self.avgpool(out)
        out = out.view(out.size(0), out.size(1))


        return out

    def forward(self, *args):
        feats = torch.stack(
            [self.extract_feat(subarea) for subarea in args], dim=0
        )

        d01 = F.pairwise_distance(feats[0], feats[1], keepdim=True)
        d02 = F.pairwise_distance(feats[0], feats[2], keepdim=True)
        d03 = F.pairwise_distance(feats[0], feats[3], keepdim=True)
        d12 = F.pairwise_distance(feats[1], feats[2], keepdim=True)
        d13 = F.pairwise_distance(feats[1], feats[3], keepdim=True)
        d23 = F.pairwise_distance(feats[2], feats[3], keepdim=True)

        mean_euclidean_distance = (d01 + d02 + d03 + d12 + d13 + d23) / 6.0


        if feats.shape[0] == 1:
            final_feat = feats.squeeze(dim=0)
        else:
            feats_mean = feats.mean(dim=0)
            feats_var = feats.var(dim=0)
            feats_min, _ = feats.min(dim=0)
            feats_max, _ = feats.max(dim=0)

            final_feat = torch.cat(
                [mean_euclidean_distance, feats_mean, feats_var, feats_min, feats_max], dim=-1
            )

        out = self.dropout(final_feat)
        out = self.fc(out)

        return out, feats[0], feats[1]

## YeNet

### Blocks

In [21]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, \
                 stride=1, with_bn=False):
        super(ConvBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, \
                              stride)
        self.relu = nn.ReLU()
        self.with_bn = with_bn
        if with_bn:
            self.norm = nn.BatchNorm2d(out_channels)
        else:
            self.norm = lambda x: x
        self.reset_parameters()

    def forward(self, x):
        return self.norm(self.relu(self.conv(x)))

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.conv.weight)
        self.conv.bias.data.fill_(0.2)
        if self.with_bn:
            self.norm.reset_parameters()


### YeNet

In [22]:
class YeNet(nn.Module):
    def __init__(self, with_bn=False, threshold=3):
        super(YeNet, self).__init__()
        self.with_bn = with_bn
        self.preprocessing = SRM_conv2d(1, 0)
        self.TLU = nn.Hardtanh(-threshold, threshold, True)
        if with_bn:
            self.norm1 = nn.BatchNorm2d(30)
        else:
            self.norm1 = lambda x: x
        self.block2 = ConvBlock(30, 30, 3, with_bn=self.with_bn)
        self.block3 = ConvBlock(30, 30, 3, with_bn=self.with_bn)
        self.block4 = ConvBlock(30, 30, 3, with_bn=self.with_bn)
        self.pool1 = nn.AvgPool2d(2, 2)
        self.block5 = ConvBlock(30, 32, 5, with_bn=self.with_bn)
        self.pool2 = nn.AvgPool2d(3, 2)
        self.block6 = ConvBlock(32, 32, 5, with_bn=self.with_bn)
        self.pool3 = nn.AvgPool2d(3, 2)
        self.block7 = ConvBlock(32, 32, 5, with_bn=self.with_bn)
        self.pool4 = nn.AvgPool2d(3, 2)
        self.block8 = ConvBlock(32, 16, 3, with_bn=self.with_bn)
        self.block9 = ConvBlock(16, 16, 3, 3, with_bn=self.with_bn)
        self.ip1 = nn.Linear(3 * 3 * 16, 2)
        self.reset_parameters()

    def forward(self, x):
        x = x.float()
        x = self.preprocessing(x)
        x = self.TLU(x)
        x = self.norm1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.pool1(x)
        x = self.block5(x)
        x = self.pool2(x)
        x = self.block6(x)
        x = self.pool3(x)
        x = self.block7(x)
        x = self.pool4(x)
        x = self.block8(x)
        x = self.block9(x)
        x = x.view(x.size(0), -1)
        x = self.ip1(x)
        return x

    def reset_parameters(self):
        for mod in self.modules():
            if isinstance(mod, SRM_conv2d) or \
                    isinstance(mod, nn.BatchNorm2d) or \
                    isinstance(mod, ConvBlock):
                mod.reset_parameters()
            elif isinstance(mod, nn.Linear):
                nn.init.normal_(mod.weight, 0. ,0.01)
                mod.bias.data.zero_()

def accuracy(outputs, labels):
    _, argmax = torch.max(outputs, 1)
    return (labels == argmax.squeeze()).float().mean()


# [MODULE] Data

## Transform

In [23]:
class ToNumpy(object):
    """Преобразует PIL Image в numpy array"""
    def __call__(self, pic):
        return np.array(pic)

class RandomRot(object):
    def __call__(self, sample):
        rot = random.randint(0, 3)

        # Проверяем размерность массива
        if sample.ndim == 2:
            # Для 2D изображения (H, W)
            return np.rot90(sample, rot).copy()
        elif sample.ndim == 3:
            # Для 3D изображения (C, H, W) или (H, W, C)
            # Предполагаем, что оси для вращения - последние две (H, W)
            return np.rot90(sample, rot, axes=(-2, -1)).copy()
        else:
            raise ValueError(f"Unexpected array dimension: {sample.ndim}")

class RandomFlip(object):
    def __init__(self, p=0.5):
        self._p = p

    def __call__(self, sample):
        if random.random() < self._p:
            if sample.ndim == 2:
                # Для 2D изображения (H, W) - отражаем по ширине (ось 1)
                return np.flip(sample, axis=1).copy()
            elif sample.ndim == 3:
                # Для 3D изображения (C, H, W) - отражаем по ширине (ось 2)
                # Или для (H, W, C) - отражаем по ширине (ось 1)
                # Предполагаем формат (C, H, W)
                return np.flip(sample, axis=2).copy()
            else:
                raise ValueError(f"Unexpected array dimension: {sample.ndim}")
        else:
            return sample


In [24]:
class RandomRot_A(A.ImageOnlyTransform):
    """Случайный поворот на 0/90/180/270 градусов"""
    def __init__(self, always_apply=False, p=1.0):
        super().__init__(always_apply, p)

    def apply(self, img, **params):
        rot = random.randint(0, 3)
        if rot == 0:
            return img
        # cv2: ROTATE_90_CLOCKWISE, ROTATE_180, ROTATE_90_COUNTERCLOCKWISE
        rotate_codes = [cv2.ROTATE_90_CLOCKWISE, cv2.ROTATE_180, cv2.ROTATE_90_COUNTERCLOCKWISE]
        rotated = cv2.rotate(img, rotate_codes[rot - 1])
        # cv2.rotate может убрать последнюю ось у (H,W,1), вернём её
        if rotated.ndim == 2:
            rotated = np.expand_dims(rotated, axis=-1)
        return rotated

    def get_transform_init_args_names(self):
        return ()


class RandomFlip_A(A.ImageOnlyTransform):
    """Горизонтальное отражение с вероятностью p"""
    def __init__(self, p=0.5, always_apply=False):
        super().__init__(always_apply, p)

    def apply(self, img, **params):
        flipped = cv2.flip(img, 1)  # 1 = horizontal flip
        if flipped.ndim == 2:
            flipped = np.expand_dims(flipped, axis=-1)
        return flipped

    def get_transform_init_args_names(self):
        return ()

## DataSet

In [25]:
file_extention_list = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.pgm')

class StegoDataset(Dataset):
    def __init__(
        self,
        cover_dir,
        stego_dir,
        num_samples=None,
        transform=None,
        shuffle=False,
        io_interface: Literal['cv2', 'pil'] = 'pil',
    ):
        self.cover_dir = cover_dir
        self.stego_dir = stego_dir
        self.transform = transform
        self.io_interface = io_interface

        stego_files = [f for f in os.listdir(stego_dir) if f.lower().endswith(file_extention_list)]
        stego_files = sorted(stego_files)
        if num_samples is not None:
            stego_files = stego_files[:num_samples]

        cover_files = [f for f in os.listdir(cover_dir) if f.lower().endswith(file_extention_list)]
        cover_files = sorted(cover_files)
        if num_samples is not None:
            cover_files = cover_files[:num_samples]

        print(f"Найдено {len(stego_files)} stego файлов и {len(cover_files)} cover файлов")

        self.files = []
        self.labels = []

        for s, c in zip(stego_files, cover_files):
            self.files.append(os.path.join(stego_dir, s))
            self.labels.append(1)

            self.files.append(os.path.join(cover_dir, s))
            self.labels.append(0)


        print(f"Загружено {len(self.files)} изображений")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        image_path = self.files[idx]
        label = self.labels[idx]

        if self.io_interface == 'cv2':
            image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
            image = image[:, :, np.newaxis]

            if self.transform:
                augmented = self.transform(image=image)
                image = augmented['image']
        else:
            image = Image.open(image_path).convert('L')

            if self.transform:
                image = self.transform(image)


        label = torch.tensor(label, dtype=torch.long)
        sample = {'images': image, 'labels': label, 'image_path': image_path}

        return sample

class WrappedConcatDataset(ConcatDataset):
    @property
    def labels(self):
        labels = []
        for dataset in self.datasets:
            if hasattr(dataset, 'labels'):
                labels.extend(dataset.labels)
        return labels


class StegoSubset(Subset):
    def __init__(self, dataset, indices):
        self.labels = [
            dataset.labels[i] for i in indices
        ]
        super().__init__(dataset, indices)


def train_val_test_split_w_shuffle(dataset, train_size, val_size, test_size, copies=1, shuffle=True, seed=42):
    ###### Проверки ########################################
    sm = round(train_size + val_size + test_size, 2)
    if sm != 1:
        raise ValueError(f'Сумма train_size, val_size, test_size != 1 ({sm:.2f})')

    all_samples = len(dataset)

    # Проверяем, что базовый датасет имеет четное количество элементов (пары cover/stego)
    if all_samples % 2 != 0:
        raise ValueError(f'Количество сэмплов должно быть чётным (пары cover/stego), got {all_samples}')

    # Вычисляем количество уникальных пар в исходном датасете
    # (Предполагаем, что dataset содержит уникальные пары, а копии создаем виртуально индексами)
    num_pairs = all_samples // 2

    # Индексы уникальных пар: [0, 1, 2, ..., num_pairs-1]
    pair_indices = list(range(num_pairs))

    if shuffle:
        rng = random.Random(seed)
        rng.shuffle(pair_indices)

    # Разбиваем УНИКАЛЬНЫЕ пары на train/val/test
    train_pairs = int(num_pairs * train_size)
    val_pairs = int(num_pairs * val_size)

    train_pair_indices = pair_indices[:train_pairs]
    val_pair_indices = pair_indices[train_pairs:train_pairs + val_pairs]
    test_pair_indices = pair_indices[train_pairs + val_pairs:]

    # Функция для развертывания индексов пар в индексы сэмплов с учетом копирования
    def pairs_to_sample_indices(pairs, duplicate_k):
        indices = []
        for p in pairs:
            # Вычисляем базовые индексы cover и stego
            c_idx = 2 * p      # cover
            s_idx = 2 * p + 1  # stego

            # Добавляем copies раз cover, затем copies раз stego
            # Пример для copies=2: c1, c1, s1, s1
            for _ in range(duplicate_k):
                indices.append(c_idx)

            for _ in range(duplicate_k):
                indices.append(s_idx)

        return indices

    # Генерируем итоговые списки индексов
    train_indices = pairs_to_sample_indices(train_pair_indices, duplicate_k=copies)
    # Для валидации и теста нет смысла размножать выборку:
    val_indices = pairs_to_sample_indices(val_pair_indices, duplicate_k=1)
    test_indices = pairs_to_sample_indices(test_pair_indices, duplicate_k=1)

    # Создаем подвыборки
    # StegoSubset просто будет брать элементы по этим индексам, создавая эффект копирования
    train_dataset = StegoSubset(dataset, train_indices)
    val_dataset = StegoSubset(dataset, val_indices)
    test_dataset = StegoSubset(dataset, test_indices)

    return train_dataset, val_dataset, test_dataset


In [26]:
def get_transform(with_aug, io_interface):
    if io_interface == 'pil':
        if with_aug:
            return transforms.Compose([
                ToNumpy(),
                RandomRot(),
                RandomFlip(),
                transforms.ToTensor(),
            ])
        else:
            return transforms.Compose([
                ToNumpy(),
                transforms.ToTensor(),
            ])
    elif io_interface == 'cv2':
        if with_aug:
            return A.Compose([
                RandomRot_A(p=1.0),
                RandomFlip_A(p=0.5),
                A.ToFloat(max_value=255.0),
                ToTensorV2(),
            ])
        else:
            return A.Compose([
                A.ToFloat(max_value=255.0),
                ToTensorV2(),
            ])

def sync_dirs(dataset, param_folder_name, copy_to_local_storage):
    # Drive:
    DRIVE_ROOT = '/content/drive/MyDrive/diplom/images'
    DRIVE_DATASET = join(DRIVE_ROOT, dataset)
    DRIVE_COVER   = join(DRIVE_DATASET, 'cover')
    DRIVE_PARAM   = join(DRIVE_DATASET, param_folder_name)
    DRIVE_PARAM_COVER = join(DRIVE_PARAM, 'cover')
    DRIVE_PARAM_STEGO = join(DRIVE_PARAM, 'stego')

    DRIVE_DATASET_cover_zip = join(DRIVE_DATASET, 'cover.zip')
    DRIVE_PARAM_cover_zip = join(DRIVE_PARAM, 'cover.zip')
    DRIVE_PARAM_stego_zip = join(DRIVE_PARAM, 'stego.zip')

    os.makedirs(DRIVE_PARAM_COVER, exist_ok=True)
    os.makedirs(DRIVE_PARAM_STEGO, exist_ok=True)

    # Local:
    LOCAL_ROOT = '/content/local_data'
    LOCAL_DATASET     = join(LOCAL_ROOT, dataset)
    LOCAL_COVER       = join(LOCAL_DATASET, 'cover')
    LOCAL_PARAM       = join(LOCAL_DATASET, param_folder_name)
    LOCAL_PARAM_COVER = join(LOCAL_PARAM, 'cover')
    LOCAL_PARAM_STEGO = join(LOCAL_PARAM, 'stego')

    LOCAL_DATASET_cover_zip = join(LOCAL_DATASET, 'cover.zip')
    LOCAL_PARAM_cover_zip = join(LOCAL_PARAM, 'cover.zip')
    LOCAL_PARAM_stego_zip = join(LOCAL_PARAM, 'stego.zip')

    os.makedirs(LOCAL_PARAM_COVER, exist_ok=True)
    os.makedirs(LOCAL_PARAM_STEGO, exist_ok=True)

    # Копируем архив с стего и ковер на локальную машину:
    if copy_to_local_storage:
        if len(os.listdir(LOCAL_PARAM_COVER)) == 0:
            if not os.path.exists(DRIVE_PARAM_cover_zip):
                cmd_zip(DRIVE_PARAM_COVER, DRIVE_PARAM_cover_zip, verbose=True)
            if not os.path.exists(LOCAL_PARAM_cover_zip):
                cmd_cp(DRIVE_PARAM_cover_zip, LOCAL_PARAM_cover_zip, verbose=True)
            if os.path.exists(DRIVE_PARAM_cover_zip):
                cmd_rm_rf(DRIVE_PARAM_cover_zip, verbose=True)

            cmd_unzip(LOCAL_PARAM_COVER, LOCAL_PARAM_cover_zip, verbose=True)
            cmd_rm_rf(LOCAL_PARAM_cover_zip, verbose=True)

        if len(os.listdir(LOCAL_PARAM_STEGO)) == 0:
            if not os.path.exists(DRIVE_PARAM_stego_zip):
                cmd_zip(DRIVE_PARAM_STEGO, DRIVE_PARAM_stego_zip, verbose=True)
            if not os.path.exists(LOCAL_PARAM_stego_zip):
                cmd_cp(DRIVE_PARAM_stego_zip, LOCAL_PARAM_stego_zip, verbose=True)
            if os.path.exists(DRIVE_PARAM_stego_zip):
                cmd_rm_rf(DRIVE_PARAM_stego_zip, verbose=True)

            cmd_unzip(LOCAL_PARAM_STEGO, LOCAL_PARAM_stego_zip, verbose=True)
            cmd_rm_rf(LOCAL_PARAM_stego_zip, verbose=True)

    if copy_to_local_storage:
        cover_dir = LOCAL_PARAM_COVER
        stego_dir = LOCAL_PARAM_STEGO
    else:
        cover_dir = DRIVE_PARAM_COVER
        stego_dir = DRIVE_PARAM_STEGO

    return cover_dir, stego_dir

def parse_config(dataset_config_list, copy_to_local_storage=False):
    dataset_list = []
    for config in dataset_config_list:
        dataset = config['image_dataset']
        h, w = config.get('hw', (256, 256))
        image_resize_strategy = config['image_resize_strategy']
        steganography_algorithm = config['steganography_algorithm']
        bpp = config['bpp']
        with_aug = config['with_aug']
        num_samples = config['num_samples']

        print('\n\t' + f'{dataset}, {bpp}bpp, num_samples: {num_samples}' )

        transform = get_transform(with_aug, io_interface)

        param_folder_name = f'{h}_{w}_{image_resize_strategy}_{steganography_algorithm}_{bpp}bpp'

        cover_dir, stego_dir = sync_dirs(dataset, param_folder_name, copy_to_local_storage)

        ds = StegoDataset(
            cover_dir=cover_dir,
            stego_dir=stego_dir,
            num_samples=num_samples,
            transform=transform,
            io_interface=io_interface,
        )
        dataset_list.append(ds)


    dataset = WrappedConcatDataset(dataset_list)
    return dataset

## DataLoader

In [27]:
from torch.utils.data import BatchSampler
from torch.utils.data import DataLoader
from torch.utils.data import Sampler
from torch.utils.data import SequentialSampler

import itertools

class TrainingSampler(Sampler):
    def __init__(self, size, seed=None, shuffle=True):
        self._size = size
        self._shuffle = shuffle
        self._seed = seed

    def __iter__(self):
        # Генерируем индексы ТОЛЬКО ОДИН раз за эпоху
        g = torch.Generator()
        g.manual_seed(self._seed)

        if self._shuffle:
            indices = torch.randperm(self._size, generator=g)
        else:
            indices = torch.arange(self._size)

        yield from indices.tolist()  # ← конечный итератор

    def __len__(self):
        return self._size

class BalancedBatchSampler(BatchSampler):

    def __init__(self, sampler, group_ids, batch_size):
        """
        Args:
            sampler (Sampler): Base sampler.
            group_ids (list[int]): If the sampler produces indices in range [0, N),
                `group_ids` must be a list of `N` ints which contains the group id of each
                sample. The group ids must be a set of integers in [0, num_groups).
            batch_size (int): Size of mini-batch.
        """
        if not isinstance(sampler, Sampler):
            raise ValueError("sampler should be an instance of torch.utils.data.Sampler, "
                             "but got sampler={}".format(sampler))

        self._sampler = sampler
        self._group_ids = np.asarray(group_ids)
        assert self._group_ids.ndim == 1
        self._batch_size = batch_size
        groups = np.unique(self._group_ids).tolist()
        assert batch_size % len(groups) == 0

        # buffer the indices of each group until batch size is reached
        self._buffer_per_group = {k: [] for k in groups}
        self._group_size = batch_size // len(groups)

    def __iter__(self):
        for idx in self._sampler:
            group_id = self._group_ids[idx]
            self._buffer_per_group[group_id].append(idx)
            if all(len(v) >= self._group_size for k, v in self._buffer_per_group.items()):
                idxs = []
                # Collect across all groups
                for k, v in self._buffer_per_group.items():
                    idxs.extend(v[:self._group_size])
                    del v[:self._group_size]

                idxs = np.random.permutation(idxs)
                yield idxs

    def __len__(self):
        return  len(self._group_ids) // self._batch_size


class TrainDataLoader(DataLoader):
    def __init__(self, dataset, batch_size, **kwargs):
        size = len(dataset)
        train_labels = dataset.labels
        sampler = TrainingSampler(
            size,
            seed=seed,
            shuffle=kwargs.get('shuffle')
        )
        batch_sampler = BalancedBatchSampler(
            sampler,
            train_labels,
            batch_size
        )

        filtered_kwargs = {
            k: v for k, v in kwargs.items()
            if k not in ['shuffle','batch_size','drop_last']
        }

        train_loader = super().__init__(
            dataset=dataset,
            batch_sampler=batch_sampler,
            **filtered_kwargs,
        )


# [MODULE] Utils

In [28]:
def load_checkpoint(checkpoint_dir, model_name, ID):
    checkpoint_path = f'{checkpoint_dir}/{model_name}_ID{ID}.pth.tar'
    if os.path.isfile(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, weights_only=False)
        return checkpoint

def save_checkpoint(state, is_best, checkpoints_dir, filename='checkpoint.pth.tar'):
    path = join(checkpoints_dir, filename)
    torch.save(state, path)
    if is_best:
        shutil.copyfile(path, join(checkpoints_dir, f'{filename}_best.pth.tar'))

def load_and_save_best_checkpoint(checkpoint_dir, checkpoint_name, checkpoint_model_name, hp, parent_id=None):
    checkpoint_path = join(checkpoint_dir, checkpoint_name)
    if os.path.isfile(checkpoint_path):
        print(f"Загрузка чекпоинта из {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, weights_only=False)
    else:
        raise ValueError(f'Чекпоинта по такому пути не существует: {checkpoint_path}')

    checkpoint = checkpoint | hp

    files = os.listdir(checkpoint_dir)
    pattern = r'_ID(\d+)'

    matching_files = [f for f in files if re.search(pattern, f)]
    id_list = []

    for f in matching_files:
        match = re.search(pattern, f)
        if match:
            id_list.append(int(match.group(1)))

    if id_list is not None and len(id_list) > 0:
        max_id = max(id_list)
        ID = max_id + 1
    else:
        ID = 1

    checkpoint['current_id'] = ID
    checkpoint['parent_id'] = parent_id

    torch.save(
        checkpoint,
        join(checkpoint_dir, f'{checkpoint_model_name}_ID{ID}.pth.tar')
    )
    print(f'Чекпоинт c ID({ID}) успешно сохранен')



In [29]:
def get_epoch_exp_data(checkpoint):
    train_metrics_dict = checkpoint.get('train_metrics_dict')
    epoch_exp_data = (
        train_metrics_dict.get('epoch_exp_data')
        if 'epoch_exp_data' in train_metrics_dict.keys()
        else train_metrics_dict
    )
    return epoch_exp_data

def get_checkpoint_chain(checkpoint_dir, model_name, start_id):
    chain = []
    current_id = start_id
    while current_id is not None:
        checkpoint = load_checkpoint(checkpoint_dir, model_name, current_id)  # предполагается, что функция определена
        chain.append(checkpoint)
        current_id = checkpoint.get('parent_id')
    chain.reverse()
    return chain

def moving_average(data, window):
    """Скользящее среднее с центрированием."""
    if window <= 1 or window > len(data):
        return data
    kernel = np.ones(window) / window
    smoothed = np.convolve(data, kernel, mode='valid')
    pad = (window - 1) // 2
    smoothed = np.pad(smoothed, (pad, window - 1 - pad), mode='edge')
    return smoothed.tolist()

def collect_chain_data(checkpoint_dir, model_name, start_id, smooth_window):
    chain = get_checkpoint_chain(checkpoint_dir, model_name, start_id)
    if not chain:
        return None

    list_train_acc     = []
    list_val_acc       = []
    list_train_loss    = []
    list_val_loss      = []
    list_learning_rate = []
    list_current_id    = []
    list_batch_size    = []
    list_samples_size  = []
    list_execution_time_sec = []

    epoch_ranges = []
    current_offset = 0

    for ckpt in chain:
        exp_data = get_epoch_exp_data(ckpt)

        train_acc     = exp_data['list_train_acc']
        val_acc       = exp_data['list_val_acc']
        train_loss    = exp_data['list_train_loss']
        val_loss      = exp_data['list_val_loss']
        learning_rate = exp_data['list_lr']
        n_epochs      = len(train_acc)
        current_id    = ckpt['current_id']
        batch_size    = ckpt['dataloader']['config']['batch_size']
        batch_size = [batch_size] * n_epochs
        execution_time_sec = ckpt['train_metrics_dict']['execution_time_sec']

        dataset_config_list = ckpt['dataset']['dataset_config_list']

        samples_size = 0
        for config in dataset_config_list:
            samples_size += config['num_samples']

        # Сглаживание
        if smooth_window > 1:
            train_acc  = moving_average(train_acc, smooth_window)
            val_acc    = moving_average(val_acc, smooth_window)
            train_loss = moving_average(train_loss, smooth_window)
            val_loss   = moving_average(val_loss, smooth_window)

        list_train_acc.extend(train_acc)
        list_val_acc.extend(val_acc)
        list_train_loss.extend(train_loss)
        list_val_loss.extend(val_loss)
        list_learning_rate.extend(learning_rate)
        list_batch_size.extend(batch_size)

        list_execution_time_sec.append(execution_time_sec)
        list_samples_size.append(samples_size)
        list_current_id.append(current_id)

        start_global = current_offset + 1
        end_global = current_offset + n_epochs
        epoch_ranges.append((start_global, end_global))
        current_offset += n_epochs

    total_epochs = current_offset
    epoch = list(range(1, total_epochs + 1))

    return {
        'epoch': epoch,
        'train_acc': list_train_acc,
        'val_acc': list_val_acc,
        'train_loss': list_train_loss,
        'val_loss': list_val_loss,
        'learning_rate': list_learning_rate,
        'current_id': list_current_id,
        'batch_size': list_batch_size,
        'epoch_ranges': epoch_ranges,
        'samples_size': list_samples_size,
        'execution_time_sec': list_execution_time_sec,
        'chain': chain,
    }

colors = [
    'rgba(255,182,193,0.3)',
    'rgba(173,216,230,0.3)',
    'rgba(144,238,144,0.3)',
    'rgba(255,255,153,0.3)',
    'rgba(221,160,221,0.3)'
]

def fm(num):
    if not isinstance(num, (int, float)):
        return str(num)

    abs_num = abs(num)

    if abs_num >= 1_000_000_000:
        return f"{num / 1_000_000_000:.0f}B"
    elif abs_num >= 1_000_000:
        return f"{num / 1_000_000:.0f}M"
    elif abs_num >= 1_000:
        return f"{num / 1_000:.0f}K"
    else:
        return str(num)

def add_time_annotations(fig, data, seconds_interval=None, row=1, col=1):
    chain = data['chain']
    epoch_ranges = data['epoch_ranges']

    # Текущее суммарное время обучения
    current_time_sec = 0.0

    for ckpt_idx, ckpt in enumerate(chain):
        # Получаем данные о времени
        # Предполагаем, что execution_time_sec - это одно число (float) для всего чекпоинта
        total_ckpt_time = ckpt['train_metrics_dict'].get('execution_time_sec', 0)

        # Получаем количество эпох в этом чекпоинте
        start_range, end_range = epoch_ranges[ckpt_idx]
        n_epochs = end_range - start_range + 1

        if n_epochs == 0:
            continue

        # Считаем среднее время одной эпохи
        time_per_epoch = total_ckpt_time / n_epochs

        # Проходим по эпохам внутри чекпоинта
        for i in range(n_epochs):
            # Время начала текущей эпохи
            epoch_start_time = current_time_sec
            # Время конца текущей эпохи
            epoch_end_time = current_time_sec + time_per_epoch

            # Номер текущего интервала (например, 0 для 0-3600, 1 для 3600-7200)
            current_interval_idx = int(epoch_start_time // seconds_interval)
            # Время конца текущего интервала
            current_interval_end = (current_interval_idx + 1) * seconds_interval

            # Проверяем, пересекает ли текущая эпоха границу интервала
            if epoch_end_time > current_interval_end:
                # Находим, какая часть эпохи прошла до границы
                time_to_boundary = current_interval_end - epoch_start_time
                fraction_of_epoch = time_to_boundary / time_per_epoch

                # Глобальный номер эпохи
                current_global_ep = start_range + i

                # Рисуем линию
                fig.add_vline(
                    x=current_global_ep + fraction_of_epoch,
                    line_width=2,
                    line_dash="dash",
                    line_color="red",
                    opacity=0.7,
                    row=row, col=1
                )

                # Добавляем аннотацию
                hours_passed = current_interval_end / 3600.0
                fig.add_annotation(
                    x=current_global_ep + fraction_of_epoch,
                    y=1,
                    yref="y domain",
                    text=f"{hours_passed:.2f}h",
                    showarrow=False,
                    yshift=5,
                    font=dict(size=10, color="darkred"),
                    bgcolor="rgba(255,255,255,0.8)",
                    row=row, col=1
                )

            # Обновляем счетчик времени
            current_time_sec += time_per_epoch

def plot_training_chain(checkpoint_dir, model_name, start_id,
                        w=1,
                        seconds_interval=1_800,
                        show_id=True
                        ):
    data = collect_chain_data(checkpoint_dir, model_name, start_id, w)
    if data is None:
        print("Чекпоинты не найдены.")
        return

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=("Loss", "Accuracy"),
        specs=[[{"secondary_y": True}], [{"secondary_y": True}]]
    )
    x = data['epoch']

    # --- Loss ---
    fig.add_trace(
        go.Scatter(x=x, y=data['train_loss'], name="Train Loss",
                    mode="lines", line=dict(color='blue')),
        row=1,col=1, secondary_y=False
    )
    fig.add_trace(
        go.Scatter(x=x, y=data['val_loss'], name="Val Loss",
                    mode="lines", line=dict(color='orange')),
        row=1, col=1, secondary_y=False
    )
    fig.add_trace(
        go.Scatter(x=x, y=data['learning_rate'], name="Learning Rate",
                    mode="lines", line=dict(color='green', dash='dash')),
        row=1, col=1, secondary_y=True
    )

    # --- Accuracy ---
    fig.add_trace(
        go.Scatter(x=x, y=data['train_acc'], name="Train Accuracy",
                    mode="lines", line=dict(color='red'), showlegend=True),
        row=2, col=1, secondary_y=False
    )
    fig.add_trace(
        go.Scatter(x=x, y=data['val_acc'], name="Val Accuracy",
                    mode="lines", line=dict(color='blue'), showlegend=True),
        row=2, col=1, secondary_y=False
    )
    fig.add_trace(
        go.Scatter(x=x, y=data['batch_size'], name="Batch size",
                    mode="lines", line=dict(color='darkgreen', dash='dash')),
        row=2, col=1, secondary_y=True
    )

    # --- Annotation: Checkpoints (ID) ---
    if show_id:
        for i, (start, end) in enumerate(data['epoch_ranges']):
            for row in [1, 2]:
                annotation_text = ''
                if row == 1:
                    annotation_text += f"ID {data['current_id'][i]}"
                    annotation_text += f"<br>{fm(data['samples_size'][i])} img"

                fig.add_vrect(
                    x0=start, x1=end,
                    fillcolor=colors[i % len(colors)],
                    opacity=0.5, layer='below', line_width=0,
                    annotation_text=annotation_text,
                    annotation_position="top left",
                    row=row, col=1
                )

    # --- Annotation: Time (Hours) ---
    if seconds_interval is not None:
        add_time_annotations(fig, data, seconds_interval=seconds_interval, row=1, col=1)
        add_time_annotations(fig, data, seconds_interval=seconds_interval, row=2, col=1)

    # --- Titles ---
    title_suffix = (
        f" (smoothed, window={w})"
        if w > 1
        else ""
    )
    # title = f"Training Metrics – chain from ID={data['current_id'][0]} to ID={data['current_id'][-1]}"
    # title += title_suffix

    title = 'Зависимость метрик обучения от эпохи'

    fig.update_layout(
        title=title,
        hovermode="x unified", plot_bgcolor='white', height=600, width=1000
    )

    # Настройка осей
    fig.update_yaxes(title_text="Loss", row=1, col=1, secondary_y=False, gridcolor='lightgray')
    fig.update_yaxes(title_text="Learning Rate", row=1, col=1, secondary_y=True, type='log', gridcolor=None)
    fig.update_yaxes(title_text="Accuracy", row=2, col=1, gridcolor='lightgray')
    fig.update_yaxes(title_text="Batch size", row=2, col=1, secondary_y=True, gridcolor=None)
    fig.update_xaxes(title_text="Epoch (cumulative)", row=2, col=1)

    fig.show()


def prepare_checkpoint_dataloader(c):
    dataset_config_list = c['dataset']['dataset_config_list']
    dataset = parse_config(dataset_config_list)

    dataloader_config = c['dataloader']['config']

    train_dataset, val_dataset, test_dataset = train_val_test_split_w_shuffle(
        dataset,
        dataloader_config['train_size'],
        dataloader_config['val_size'],
        dataloader_config['test_size'],
        shuffle=dataloader_config['dataset_shuffle'],
        seed=dataloader_config['seed'],
    )

    test_loader = DataLoader(
        test_dataset,
        dataloader_config['batch_size'],
        **c['dataloader']['test_loader_kwargs']
    )
    return test_loader



In [30]:
class Trainer:
    def __init__(
            self,
            net: nn.Module,
            criterion_1=None,
            criterion_2=None,
            optimizer=None,
            scheduler=None,
            scaler=None,
            hp: Dict=None,
            subarea_out_num: int=2
        ):
        self.net = net
        self.criterion_1 = criterion_1
        self.criterion_2 = criterion_2
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.scaler = scaler
        self.subarea_out_num = subarea_out_num

        if hp is None:
            self.hp = dict()
            self.alpha = None
        else:
            self.hp = hp
            self.alpha = self.hp['train_config']['alpha']
        self.epoch_exp_data = None

    @staticmethod
    def _preprocess_data(images, labels, random_crop=False, subarea_out_num=2):
        # images of shape: NxCxHxW
        if images.ndim == 5:  # 1xNxCxHxW
            images = images.squeeze(0)
            labels = labels.squeeze(0)
        h, w = images.shape[-2:]

        if random_crop:
            ch = random.randint(h * 3 // 4, h)  # h // 2      #256
            cw = random.randint(w * 3 // 4, w)  # square ch   #256

            h0 = random.randint(0, h - ch)  # 128
            w0 = random.randint(0, w - cw)  # 128
        else:
            ch, cw, h0, w0 = h, w, 0, 0


        cw = cw & ~1

        if subarea_out_num == 2:
            inputs = [
                images[..., h0:h0 + ch, w0:w0 + cw // 2],
                images[..., h0:h0 + ch, w0 + cw // 2:w0 + cw]
            ]
        elif subarea_out_num == 4:
            mid_h = h0 + ch // 2
            mid_w = w0 + cw // 2

            inputs = [
                images[..., h0:mid_h, w0:mid_w],       # Top-Left
                images[..., h0:mid_h, mid_w:w0 + cw],  # Top-Right
                images[..., mid_h:h0 + ch, w0:mid_w],  # Bottom-Left
                images[..., mid_h:h0 + ch, mid_w:w0 + cw] # Bottom-Right
            ]
        else:
            raise NotImplementedError(f'subarea_out_num should be 2 or 4, got {subarea_out_num}')

        inputs = [x.to(device) for x in inputs]
        labels = labels.to(device)

        return inputs, labels

    @staticmethod
    def _print_exp_results(checkpoint_state: dict):
        print(f"\n")
        print(f"Результаты эпохи {checkpoint_state['epoch']}:")
        print(f"Тренировка - Loss: {checkpoint_state['train_loss']:.4f}, Acc: {checkpoint_state['train_acc']:.2f}%")
        print(f"Валидация  - Loss: {checkpoint_state['val_loss']:.4f}, Acc: {checkpoint_state['val_acc']:.2f}%")
        print(f"LR = {checkpoint_state['current_lr']:.6f}")

    @staticmethod
    def _plot_roc_curve(labels, probs):
        plt.figure(figsize=(8, 6))
        labels = np.array(labels)
        probs = np.array(probs)

        fpr, tpr, _ = roc_curve(labels, probs[:, 1])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=2, label=f'Класс 1 (AUC = {roc_auc:.3f})')

        # Диагональ (случайный классификатор)
        plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Случайный (AUC=0.5)')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('ROC-кривая')
        plt.legend(loc='lower right')
        plt.grid(alpha=0.3)

        plt.show()


    def test(self, test_loader, plot_roc=True):
        self.net.eval()
        all_labels = []
        all_probs = []
        all_preds = []

        with torch.no_grad():
            progress_bar = tqdm(test_loader, desc='Тестирование')
            for batch_data in progress_bar:
                images = batch_data['images'].to(device, non_blocking=True)
                labels = batch_data['labels'].to(device, non_blocking=True)
                inputs, labels = self._preprocess_data(images, labels, subarea_out_num=self.subarea_out_num)

                outputs, _, _ = self.net(*inputs)
                probs = torch.softmax(outputs, dim=1)
                preds = outputs.argmax(dim=1)

                all_labels.extend(labels.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())

        accuracy = accuracy_score(all_labels, all_preds)

        pos_probs = [p[1] for p in all_probs]
        roc_auc = roc_auc_score(all_labels, pos_probs)

        print(f"\nРезультаты на тесте:")
        print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
        print(f"ROC AUC (macro OvR): {roc_auc:.4f}")

        # Построение ROC-кривой
        if plot_roc:
            self._plot_roc_curve(all_labels, all_probs)

        self.hp['test_metrics_dict'] = dict(
            accuracy=accuracy,
            roc_auc=roc_auc,
            raw=dict(
                all_labels=all_labels,
                all_probs=all_probs,
                pos_probs=pos_probs,
            )
        )


    def validate(self, data_loader):
        self.net.eval()
        running_loss = 0.0
        running_acc = 0.0

        with torch.no_grad():
            progress_bar = tqdm(data_loader, desc=f'Валидация')
            for batch_idx, batch_data in enumerate(progress_bar):
                images = batch_data['images'].to(device, non_blocking=True)
                labels = batch_data['labels'].to(device, non_blocking=True)
                inputs, labels = self._preprocess_data(images, labels, subarea_out_num=self.subarea_out_num)

                self.optimizer.zero_grad()
                outputs, feats_0, feats_1 = self.net(*inputs)

                predictions = outputs.argmax(dim=1)
                loss = self.criterion_1(outputs, labels) + self.alpha * self.criterion_2(feats_0, feats_1, labels)
                accuracy = (predictions == labels).float().mean().item()

                running_loss += loss.item()
                running_acc += accuracy

        val_loss = running_loss / len(data_loader)
        val_acc = running_acc / len(data_loader)

        return val_loss, val_acc


    def train(self, data_loader):
        self.net.train()
        running_loss = 0.0
        running_acc = 0.0

        progress_bar = tqdm(data_loader, desc=f'Обучение', smoothing=0.0)
        for batch_idx, batch_data in enumerate(progress_bar):
            images = batch_data['images'].to(device, non_blocking=True)
            labels = batch_data['labels'].to(device, non_blocking=True)
            inputs, labels = self._preprocess_data(images, labels, subarea_out_num=self.subarea_out_num)

            self.optimizer.zero_grad()

            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
                outputs, feats_0, feats_1 = net(*inputs)

                predictions = outputs.argmax(dim=1)
                loss = self.criterion_1(outputs, labels) + self.alpha * self.criterion_2(feats_0, feats_1, labels)
                accuracy = (predictions == labels).float().mean().item()

            running_loss += loss.item()
            running_acc += accuracy


            # Изменение весов:
            self.scaler.scale(loss).backward()
            self.scaler.step(optimizer)
            self.scaler.update()

        train_loss = running_loss / len(data_loader)
        train_acc = running_acc / len(data_loader)

        return train_loss, train_acc


    def train_val_step(self, epoch, train_loader, val_loader):

        # Train
        train_loss, train_acc = self.train(train_loader)
        self.epoch_exp_data['list_train_loss'].append(train_loss)
        self.epoch_exp_data['list_train_acc'].append(train_acc)

        # Validate
        val_loss, val_acc = self.validate(val_loader)
        self.epoch_exp_data['list_val_loss'].append(val_loss)
        self.epoch_exp_data['list_val_acc'].append(val_acc)

        self.scheduler.step()
        current_lr = self.optimizer.param_groups[0]['lr']
        self.epoch_exp_data['list_lr'].append(current_lr)

        # Result
        checkpoint_snp = dict(
            epoch = epoch+1,
            train_loss = train_loss,
            train_acc = train_acc,
            val_loss = val_loss,
            val_acc = val_acc,
            current_lr = current_lr,
        )

        return checkpoint_snp


    def train_val_iter(self, num_epochs, train_loader, val_loader,
                       checkpoint_dir,
                       checkpoint_name,
                       verbose=True):
        self.epoch_exp_data = dict(
            list_train_loss = [],
            list_train_acc  = [],
            list_val_loss   = [],
            list_val_acc    = [],
            list_lr         = [],
        )
        best_val_acc = 0

        start_time = time.time()
        for epoch in range(num_epochs):
            title(f"Эпоха {epoch+1}/{num_epochs}")

            # Train - Val - Step LR:
            checkpoint_snp = self.train_val_step(epoch, train_loader, val_loader)
            self._print_exp_results(checkpoint_snp)

            # Save checkpoint:
            is_best = checkpoint_snp['val_acc'] > best_val_acc
            if is_best:
                best_val_acc = checkpoint_snp['val_acc']
            cur_execution_time_sec = time.time() - start_time

            checkpoint_state = dict(
                checkpoint_snp = checkpoint_snp,
                best_val_acc = best_val_acc,

                state_dict = self.net.state_dict(),
                optimizer = self.optimizer.state_dict(),
                scheduler = self.scheduler.state_dict(),
                scaler = self.scaler.state_dict(),

                train_metrics_dict= dict(
                    epoch_exp_data=self.epoch_exp_data,
                    execution_time_sec=cur_execution_time_sec,
                )
            ) | self.hp

            save_checkpoint(
                checkpoint_state,
                is_best,
                checkpoints_dir=checkpoint_dir,
                filename=checkpoint_name
            )

        end_time = time.time()
        execution_time_sec = end_time - start_time

        self.hp['train_metrics_dict'] = dict(
            epoch_exp_data=self.epoch_exp_data,
            execution_time_sec=execution_time_sec,
        )


        if verbose:
            title("Обучение завершено!")
            print(f"Лучшая точность на валидации: {best_val_acc:.2f}%")
            print(f"Время выполнения: {execution_time_sec:.0f} секунд ({execution_time_sec / 60 :.2f} минут)")



# [RUN] DataSet

In [ ]:
bpp = 0.5
bpp_checkpoint = 'bpp' + str(bpp).replace('.','')
io_interface = 'pil'

copies=2
dataset_config_list = [
    dict(
        image_dataset = 'BOSSbase',
        hw=(256,256),
        image_resize_strategy = 'resize',
        steganography_algorithm = 's-uniward',
        bpp = bpp,
        with_aug = True,
        num_samples = 10_000,
    ),
    dict(
        image_dataset = 'BOWS2',
        hw=(256,256),
        image_resize_strategy = 'resize',
        steganography_algorithm = 's-uniward',
        bpp = bpp,
        with_aug = True,
        num_samples = 10_000,
    ),
]

hp['dataset'] = dict(
    dataset_config_list=dataset_config_list,
    io_interface=io_interface,
    copies=copies,
)

dataset = parse_config(dataset_config_list, copy_to_local_storage=True)

# [RUN] DataLoader

In [ ]:
################################################################################
# Гипер-параметры:
################################################################################
batch_size = 32
train_size = .6
val_size = .2
test_size = .2
seed = 42
dataset_shuffle = True


hp['dataloader'] = dict()
hp['dataloader']['config'] = dict(
    seed=seed,
    batch_size=batch_size,
    train_size=train_size,
    val_size=val_size,
    test_size=test_size,
    dataset_shuffle=dataset_shuffle,
)

train_loader_kwargs = dict(
    num_workers = 12,
    prefetch_factor = 16,
    persistent_workers = True,
    shuffle = True,
    pin_memory = True,
    drop_last = False,
)
hp['dataloader']['train_loader_kwargs'] = train_loader_kwargs

val_loader_kwargs = dict(
    num_workers = 12,
    prefetch_factor = 16,
    persistent_workers = True,
    shuffle = False,
    pin_memory = True,
    drop_last = False,
)
hp['dataloader']['val_loader_kwargs'] = val_loader_kwargs

test_loader_kwargs = dict(
    num_workers = 12,
    prefetch_factor = 16,
    persistent_workers = True,
    shuffle = False,
    pin_memory = True,
    drop_last = False,
)
hp['dataloader']['test_loader_kwargs'] = test_loader_kwargs
################################################################################

random.seed(seed)
np.random.seed(seed)
torch.set_rng_state(torch.manual_seed(seed).get_state())

train_dataset, val_dataset, test_dataset = train_val_test_split_w_shuffle(
    dataset,
    train_size,
    val_size,
    test_size,
    shuffle=dataset_shuffle,
    seed=seed,
    copies=copies,
)

train_loader = TrainDataLoader(train_dataset,batch_size,**train_loader_kwargs)
val_loader = DataLoader(val_dataset,batch_size,**val_loader_kwargs)
test_loader = DataLoader(test_dataset,batch_size,**test_loader_kwargs)

print(f"Тренировочных батчей: {len(train_loader)}")
print(f"Валидационных батчей: {len(val_loader)}")
print(f"Тестовых батчей:      {len(test_loader)}")

# [RUN] Train

In [65]:
################################################################################
# Гипер-параметры:
################################################################################
alpha = 0.1
margin = 1
eps = 1e-8
weight_decay = 1e-4

num_epochs = 120
learning_rate = 1e-3
model_p = 0.5

checkpoint_model_name='SiaStegNet_FC'
checkpoint_name=f'{checkpoint_model_name}.pth.tar'
checkpoint_dir = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/{bpp_checkpoint}'

hp['train_config'] = dict(
    alpha=alpha,
    margin=margin,
    eps=eps,
    weight_decay=weight_decay,
    num_epochs=num_epochs,
    learning_rate=learning_rate,
    model_p=model_p,
)
################################################################################


# Modules:
net = SiaStegNet_4S(p=model_p).to(device)
net.train()

criterion_1 = nn.CrossEntropyLoss().to(device)
criterion_2 = ContrastiveLoss(margin=margin).to(device)

optimizer = Adamax(
    net.parameters(),
    lr=learning_rate,
    eps=eps,
    weight_decay=weight_decay
)


def lr_lambda(epoch):
    if epoch < 100:
        return 1.0
    else:
        return 0.1

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lr_lambda
)


scaler = GradScaler(
    enabled=use_amp
)

# Обучение:
trainer = Trainer(net=net,
                  criterion_1=criterion_1,
                  criterion_2=criterion_2,
                  optimizer=optimizer,
                  scheduler=scheduler,
                  scaler=scaler,
                  hp=hp,
                  subarea_out_num=4)

trainer.train_val_iter(num_epochs,train_loader,val_loader,
                       checkpoint_dir,
                       checkpoint_name=checkpoint_name,
                       )
trainer.test(test_loader)

In [66]:
load_and_save_best_checkpoint(
    checkpoint_dir=checkpoint_dir,
    checkpoint_name=checkpoint_name,
    checkpoint_model_name=checkpoint_model_name,
    parent_id=None,
    hp=trainer.hp,
)

# [ANALYSIS] Анализ обучения

## SiaStegNet

### [ID=_] [SiaStegNet] | no train shuffle

### [ID=71] [SiaStegNet] 40k img | bs64 | no leak

In [ ]:
ID = 71
dr = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/bpp05'
model_name = 'SiaStegNet_batch_size_64_boss_bows_w_aug_double_datasets_no_leakadge'
plot_training_chain(dr, model_name, ID, show_id=True)

In [ ]:
c = torch.load(
    f'{dr}/{model_name}_ID{ID}.pth.tar',
    weights_only=False
)
test_loader = prepare_checkpoint_dataloader(c)

net = SiaStegNet(p=.5).to(device)
net.load_state_dict(c['state_dict'])
trainer = Trainer(net=net)
trainer.test(test_loader)

### [ID=65] [SiaStegNet] 40k img | bs64 | with leak

In [ ]:
ID = 65
dr = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/bpp05'
model_name = 'SiaStegNet_batch_size_64_boss_bows_w_aug_double_datasets'
plot_training_chain(dr, model_name, ID, show_id=False)

In [ ]:
c = torch.load(
    f'{dr}/{model_name}_ID{ID}.pth.tar',
    weights_only=False
)

In [ ]:
c['dataloader']

### [ID=56] [SiaStegNet] 20k img | bs32 | no aug | overfit

In [ ]:
ID = 56
dr = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/bpp05'
model_name = 'SiaStegNet_batch_size_32_boss_bows_no_aug'
plot_training_chain(dr, model_name, ID, show_id=True)

### [ID=55] [SiaStegNet] 20k | bs64 | 100+30+30

In [ ]:
ID = 55
dr = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/bpp05'
model_name = 'SiaStegNet_batch_size_64_boss_bows'
plot_training_chain(dr, model_name, ID, show_id=True)

### [ID=54] [SiaStegNet] 20k img | bs16 | 100+30

In [ ]:
ID = 54
dr = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/bpp05'
model_name = 'SiaStegNet_batch_size_16_boss_bows'
plot_training_chain(dr, model_name, ID, show_id=True)

### [ID=52] [SiaStegNet] 20k img | bs32

In [ ]:
ID = 52
dr = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/bpp05'
model_name = 'SiaStegNet_batch_size_32_boss_bows'
plot_training_chain(dr, model_name, ID, show_id=True)

### [ID=39] [SiaStegNet] 10k img | bs32

In [ ]:
ID = 39
dr = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/bpp05'
model_name = 'SiaStegNet'
plot_training_chain(dr, model_name, ID, show_id=True)

## SiaStegNet + SE

### [ID=50] [SiaStegNet + SE] 10k img | bs32

In [ ]:
ID = 50
dr = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/bpp05'
model_name = 'SiaStegNet_w_SE'
plot_training_chain(dr, model_name, ID, show_id=True)

In [ ]:
# c = load_checkpoint(dr, model_name, ID)

### [ID=57] [SiaStegNet + SE] 20k img | bs64

In [ ]:
ID = 57
dr = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/bpp05'
model_name = 'SiaStegNet_SE'
plot_training_chain(dr, model_name, ID, show_id=True)

### [ID=67] [SiaStegNet + SE] 40k img | bs64 | leak

In [ ]:
ID = 67
dr = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/bpp05'
model_name = 'SiaStegNet_SE'
plot_training_chain(dr, model_name, ID, show_id=True)

### [ID=70] [SiaStegNet + SE] 40k img | bs64

In [ ]:
ID = 70
dr = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/bpp05'
model_name = 'SiaStegNet_SE'
plot_training_chain(dr, model_name, ID, show_id=True)

In [ ]:
c = torch.load(
    f'{dr}/{model_name}_ID{ID}.pth.tar',
    weights_only=False
)

In [ ]:
print('test_acc:', c['test_metrics_dict']['accuracy'])
print('test_roc:', c['test_metrics_dict']['roc_auc'])

## SiaStegNet + CVT

### [ID=58] [SiaStegNet + CVT] 20k img | bs64 | 100+30

In [ ]:
ID = 58
dr = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/bpp05'
model_name = 'SiaStegNet_CVT'
plot_training_chain(dr, model_name, ID, show_id=True)

### [ID=69] [SiaStegNet + CVT] 40k img | bs64 | 100+20 | leak

In [ ]:
ID = 69
dr = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/bpp05'
model_name = 'SiaStegNet_CVT'
plot_training_chain(dr, model_name, ID, show_id=True)

### [ID=73] [SiaStegNet + CVT] 40k img | bs64 | 100+20 | no leak

In [ ]:
ID = 73
dr = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/bpp05'
model_name = 'SiaStegNet_CVT_no_leak'
plot_training_chain(dr, model_name, ID, show_id=False)

## SiaStegNet + FC

### [ID=76] [SiaStegNet + 4 subnets] 40k img | bs32 | 100+20

In [70]:
ID = 76
dr = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/bpp05'
model_name = 'SiaStegNet_FC'
plot_training_chain(dr, model_name, ID, show_id=False, seconds_interval=7_200)

### [ID=74] [SiaStegNet + 4 subnets] 20k img | bs64 | 100+20

In [71]:
ID = 75
dr = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/bpp05'
model_name = 'SiaStegNet_FC'
plot_training_chain(dr, model_name, ID, show_id=False, seconds_interval=3_600)

### [ID=74] [SiaStegNet + FC] 40k img | bs64 | 100+20

In [ ]:
ID = 74
dr = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/bpp05'
model_name = 'SiaStegNet_FC'
plot_training_chain(dr, model_name, ID, show_id=False)

# [RUN] Tune

In [ ]:
ID = 64
bpp = 'bpp05'
model_name = 'SiaStegNet_batch_size_64_boss_bows_w_aug_double_datasets'

tmp_checkpoint = torch.load(
    f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/{bpp}/{model_name}_ID{ID}.pth.tar',
    weights_only=False
)
state_dict = tmp_checkpoint['state_dict']
optimizer_state_dict = tmp_checkpoint['optimizer']
print(f'ID:{ID}')

In [ ]:
################################################################################
# Гипер-параметры:
################################################################################
alpha = 0.1
margin = 1
eps = 1e-8
weight_decay = 1e-4

num_epochs = 20
learning_rate = 1e-4
model_p = 0.5

model_name='SiaStegNet_batch_size_64_boss_bows_w_aug_double_datasets'
checkpoint_name=f'{model_name}.pth.tar'
checkpoint_dir = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/{bpp_checkpoint}'

hp['train_config'] = dict(
    alpha=alpha,
    margin=margin,
    eps=eps,
    weight_decay=weight_decay,
    num_epochs=num_epochs,
    learning_rate=learning_rate,
    model_p=model_p,
)
################################################################################


# Modules:
net = SiaStegNet(p=model_p).to(device)
net.load_state_dict(state_dict)
net.train()

criterion_1 = nn.CrossEntropyLoss().to(device)
criterion_2 = ContrastiveLoss(margin=margin).to(device)

optimizer = Adamax(
    net.parameters(),
    lr=learning_rate,
    eps=eps,
    weight_decay=weight_decay
)


def lr_lambda(epoch):
    if epoch < 130:
        return 1.0
    else:
        return 0.1

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    # lr_lambda=lr_lambda,
    lr_lambda=lambda epoch: 1.0
)


scaler = GradScaler(
    enabled=use_amp
)

# Обучение:
trainer = Trainer(
    net=net,
    criterion_1=criterion_1,
    criterion_2=criterion_2,
    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,
    hp=hp,
)

trainer.train_val_iter(num_epochs,train_loader,val_loader,checkpoint_dir,
                       checkpoint_name=checkpoint_name)
trainer.test(test_loader)

In [ ]:
load_and_save_best_checkpoint(
    checkpoint_dir=checkpoint_dir,
    checkpoint_name=checkpoint_name,
    model_name=model_name,
    parent_id=ID,
    hp=trainer.hp,
)

# Финальное тестирование полученных моделей

## Utils

In [31]:
def get_SiaStegNet():
    ID = 71
    dr = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/bpp05'
    model_name = 'SiaStegNet_batch_size_64_boss_bows_w_aug_double_datasets_no_leakadge'

    c = torch.load(
        f'{dr}/{model_name}_ID{ID}.pth.tar',
        weights_only=False
    )

    net = SiaStegNet(p=.5).to(device)
    net.load_state_dict(c['state_dict'])

    return net


def get_SiaStegNet_4S():
    ID = 75
    dr = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/bpp05'
    model_name = 'SiaStegNet_FC'


    c = torch.load(
        f'{dr}/{model_name}_ID{ID}.pth.tar',
        weights_only=False
    )

    net = SiaStegNet_4S(p=.5).to(device)
    net.load_state_dict(c['state_dict'])

    return net

def get_SiaStegNet_SE():
    ID = 70
    dr = f'/content/drive/MyDrive/diplom/checkpoints/SiaStegNet/bpp05'
    model_name = 'SiaStegNet_SE'


    c = torch.load(
        f'{dr}/{model_name}_ID{ID}.pth.tar',
        weights_only=False
    )

    net = SiaStegNet_SE(p=.5).to(device)
    net.load_state_dict(c['state_dict'])

    return net


In [32]:
io_interface = 'pil'
copies=1
batch_size = 64
train_size = .6
val_size = .2
test_size = .2
seed = 42
dataset_shuffle = True

random.seed(seed)
np.random.seed(seed)
torch.set_rng_state(torch.manual_seed(seed).get_state())

test_loader_kwargs = dict(
    num_workers = 12,
    prefetch_factor = 16,
    persistent_workers = True,
    shuffle = False,
    pin_memory = True,
    drop_last = False,
)

def get_test_loader(hw, algorithm, bpp, copy_to_local_storage=False):
    dataset_config_list = [
    dict(
        image_dataset = 'BOSSbase',
        hw=hw,
        image_resize_strategy = 'resize',
        steganography_algorithm = algorithm,
        bpp = bpp,
        with_aug = True,
        num_samples = 10_000,
    ),
    dict(
        image_dataset = 'BOWS2',
        hw=hw,
        image_resize_strategy = 'resize',
        steganography_algorithm = algorithm,
        bpp = bpp,
        with_aug = True,
        num_samples = 10_000,
    ),
]
    dataset = parse_config(dataset_config_list, copy_to_local_storage=copy_to_local_storage)

    train_dataset, val_dataset, test_dataset = train_val_test_split_w_shuffle(
        dataset,
        train_size,
        val_size,
        test_size,
        shuffle=dataset_shuffle,
        seed=seed,
        copies=copies,
    )

    test_loader = DataLoader(test_dataset,batch_size,**test_loader_kwargs)

    return test_loader

## 256x256

### S-UNIWARD

#### bpp=0.5

In [42]:
bpp=.5
hw=(256,256)
algorithm='s-uniward'

net = get_SiaStegNet_SE()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


#### bpp=0.4

In [36]:
bpp=.4
hw=(256,256)
algorithm='s-uniward'

net = get_SiaStegNet()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


In [37]:
bpp=.4
hw=(256,256)
algorithm='s-uniward'

net = get_SiaStegNet_4S()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net, subarea_out_num=4)
trainer.test(test_loader)


In [39]:
bpp=.4
hw=(256,256)
algorithm='s-uniward'

net = get_SiaStegNet_SE()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


#### bpp=0.3

In [45]:
bpp=.3
hw=(256,256)
algorithm='s-uniward'

net = get_SiaStegNet()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


In [47]:
bpp=.3
hw=(256,256)
algorithm='s-uniward'

net = get_SiaStegNet_4S()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net, subarea_out_num=4)
trainer.test(test_loader)


In [48]:
bpp=.3
hw=(256,256)
algorithm='s-uniward'

net = get_SiaStegNet_SE()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


#### bpp=0.2

In [49]:
bpp=.2
hw=(256,256)
algorithm='s-uniward'

net = get_SiaStegNet()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


In [50]:
bpp=.2
hw=(256,256)
algorithm='s-uniward'

net = get_SiaStegNet_4S()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net, subarea_out_num=4)
trainer.test(test_loader)


In [51]:
bpp=.2
hw=(256,256)
algorithm='s-uniward'

net = get_SiaStegNet_SE()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


### WOW

#### bpp=0.5

In [53]:
bpp=.5
hw=(256,256)
algorithm='wow'

net = get_SiaStegNet()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


In [54]:
bpp=.5
hw=(256,256)
algorithm='wow'

net = get_SiaStegNet_4S()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net, subarea_out_num=4)
trainer.test(test_loader)


In [55]:
bpp=.5
hw=(256,256)
algorithm='wow'
net = get_SiaStegNet_SE()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


#### bpp=0.4

In [56]:
bpp=.4
hw=(256,256)
algorithm='wow'

net = get_SiaStegNet()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


In [57]:
bpp=.4
hw=(256,256)
algorithm='wow'

net = get_SiaStegNet_4S()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net, subarea_out_num=4)
trainer.test(test_loader)


In [58]:
bpp=.4
hw=(256,256)
algorithm='wow'
net = get_SiaStegNet_SE()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


#### bpp=0.3

In [59]:
bpp=.3
hw=(256,256)
algorithm='wow'

net = get_SiaStegNet()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


In [60]:
bpp=.3
hw=(256,256)
algorithm='wow'

net = get_SiaStegNet_4S()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net, subarea_out_num=4)
trainer.test(test_loader)


In [61]:
bpp=.3
hw=(256,256)
algorithm='wow'
net = get_SiaStegNet_SE()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


#### bpp=0.2

In [62]:
bpp=.2
hw=(256,256)
algorithm='wow'

net = get_SiaStegNet()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


In [63]:
bpp=.2
hw=(256,256)
algorithm='wow'

net = get_SiaStegNet_4S()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net, subarea_out_num=4)
trainer.test(test_loader)


In [64]:
bpp=.2
hw=(256,256)
algorithm='wow'
net = get_SiaStegNet_SE()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


### HUGO

#### bpp=0.5

In [34]:
bpp=.5
hw=(256,256)
algorithm='hugo'

net = get_SiaStegNet()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


In [35]:
bpp=.5
hw=(256,256)
algorithm='hugo'

net = get_SiaStegNet_4S()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net, subarea_out_num=4)
trainer.test(test_loader)


In [36]:
bpp=.5
hw=(256,256)
algorithm='hugo'
net = get_SiaStegNet_SE()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


#### bpp=0.4

In [72]:
bpp=.4
hw=(256,256)
algorithm='hugo'

net = get_SiaStegNet()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


In [73]:
bpp=.4
hw=(256,256)
algorithm='hugo'

net = get_SiaStegNet_4S()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net, subarea_out_num=4)
trainer.test(test_loader)


In [74]:
bpp=.4
hw=(256,256)
algorithm='hugo'
net = get_SiaStegNet_SE()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


#### bpp=0.3

In [69]:
bpp=.3
hw=(256,256)
algorithm='hugo'

net = get_SiaStegNet()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


In [70]:
bpp=.3
hw=(256,256)
algorithm='hugo'

net = get_SiaStegNet_4S()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net, subarea_out_num=4)
trainer.test(test_loader)


In [71]:
bpp=.3
hw=(256,256)
algorithm='hugo'
net = get_SiaStegNet_SE()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


#### bpp=0.2

In [66]:
bpp=.2
hw=(256,256)
algorithm='hugo'

net = get_SiaStegNet()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


In [67]:
bpp=.2
hw=(256,256)
algorithm='hugo'

net = get_SiaStegNet_4S()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net, subarea_out_num=4)
trainer.test(test_loader)


In [68]:
bpp=.2
hw=(256,256)
algorithm='hugo'
net = get_SiaStegNet_SE()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


## 512x512

### S-UNIWARD

#### bpp=0.5

In [37]:
bpp=.5
hw=(512,512)
algorithm='s-uniward'

net = get_SiaStegNet()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


In [38]:
bpp=.5
hw=(512,512)
algorithm='s-uniward'

net = get_SiaStegNet_4S()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net, subarea_out_num=4)
trainer.test(test_loader)


In [39]:
bpp=.5
hw=(512,512)
algorithm='s-uniward'

net = get_SiaStegNet_SE()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


#### bpp=0.4

In [41]:
bpp=.4
hw=(512,512)
algorithm='s-uniward'

net = get_SiaStegNet()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


In [42]:
bpp=.4
hw=(512,512)
algorithm='s-uniward'

net = get_SiaStegNet_4S()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net, subarea_out_num=4)
trainer.test(test_loader)


In [43]:
bpp=.4
hw=(512,512)
algorithm='s-uniward'

net = get_SiaStegNet_SE()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


#### bpp=0.3

In [40]:
bpp=.3
hw=(512,512)
algorithm='s-uniward'

net = get_SiaStegNet()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


In [ ]:
bpp=.3
hw=(512,512)
algorithm='s-uniward'

net = get_SiaStegNet_4S()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net, subarea_out_num=4)
trainer.test(test_loader)


In [ ]:
bpp=.3
hw=(512,512)
algorithm='s-uniward'

net = get_SiaStegNet_SE()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


#### bpp=0.2

In [44]:
bpp=.2
hw=(512,512)
algorithm='s-uniward'

net = get_SiaStegNet()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


In [ ]:
bpp=.2
hw=(512,512)
algorithm='s-uniward'

net = get_SiaStegNet_4S()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net, subarea_out_num=4)
trainer.test(test_loader)


In [ ]:
bpp=.2
hw=(512,512)
algorithm='s-uniward'

net = get_SiaStegNet_SE()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


### WOW

#### bpp=0.5

In [45]:
bpp=.5
hw=(512,512)
algorithm='wow'

net = get_SiaStegNet()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


In [46]:
bpp=.5
hw=(512,512)
algorithm='wow'

net = get_SiaStegNet_4S()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net, subarea_out_num=4)
trainer.test(test_loader)


In [47]:
bpp=.5
hw=(512,512)
algorithm='wow'

net = get_SiaStegNet_SE()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


#### bpp=0.4

In [48]:
bpp=.4
hw=(512,512)
algorithm='wow'

net = get_SiaStegNet()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


In [49]:
bpp=.4
hw=(512,512)
algorithm='wow'

net = get_SiaStegNet_4S()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net, subarea_out_num=4)
trainer.test(test_loader)


In [51]:
bpp=.4
hw=(512,512)
algorithm='wow'

net = get_SiaStegNet_SE()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


#### bpp=0.3

In [ ]:
bpp=.3
hw=(512,512)
algorithm='wow'

net = get_SiaStegNet()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


In [ ]:
bpp=.3
hw=(512,512)
algorithm='wow'

net = get_SiaStegNet_4S()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net, subarea_out_num=4)
trainer.test(test_loader)


In [ ]:
bpp=.3
hw=(512,512)
algorithm='wow'

net = get_SiaStegNet_SE()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


#### bpp=0.2

In [ ]:
bpp=.2
hw=(512,512)
algorithm='wow'

net = get_SiaStegNet()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


In [ ]:
bpp=.2
hw=(512,512)
algorithm='wow'

net = get_SiaStegNet_4S()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net, subarea_out_num=4)
trainer.test(test_loader)


In [ ]:
bpp=.2
hw=(512,512)
algorithm='wow'

net = get_SiaStegNet_SE()
test_loader = get_test_loader(hw, algorithm, bpp)
trainer = Trainer(net=net)
trainer.test(test_loader)


### HUGO

#### bpp=0.5

#### bpp=0.4

#### bpp=0.3

#### bpp=0.2